# C11-neural-training — Practice p15 — Solution


**Type:** integrative (parts consume earlier results) · **Difficulty:** core · **Concepts:** manual-backpropagation, trained-mlp


The helper returns the full forward snapshot and all four gradients; the
training loop computes the whole gradient dictionary before simultaneous
replacement of the parameters.


In [ ]:
import numpy as np

def _numpy_mlp_gradients(X,y,params):
    z1=X@params["W1"].T+params["b1"]; h=np.maximum(z1,0); scores=h@params["W2"].T+params["b2"]
    shifted=scores-scores.max(1,keepdims=True); exp_s=np.exp(shifted); p=exp_s/exp_s.sum(1,keepdims=True)
    loss=float(np.mean(np.log(exp_s.sum(1))-shifted[np.arange(len(y)),y]))
    ds=p.copy(); ds[np.arange(len(y)),y]-=1; ds/=len(y)
    grads={"W2":ds.T@h,"b2":ds.sum(0)}; dz1=(ds@params["W2"])*(z1>0)
    grads.update(W1=dz1.T@X,b1=dz1.sum(0))
    return loss,{k:grads[k] for k in ("W1","b1","W2","b2")}

def train_numpy_mlp(seed=20260804,epochs=800,learning_rate=0.12):
    rng=np.random.default_rng(seed); centers=np.array([[-1.,-1.],[-1.,1.],[1.,-1.],[1.,1.]])
    clean_labels=np.array([0,1,1,0]); X=np.repeat(centers,24,axis=0)+.12*rng.standard_normal((96,2)); y=np.repeat(clean_labels,24)
    params={"W1":rng.standard_normal((8,2))*np.sqrt(1/2),"b1":np.zeros(8),
            "W2":rng.standard_normal((2,8))*np.sqrt(1/8),"b2":np.zeros(2)}
    initial={k:v.copy() for k,v in params.items()}; losses=np.empty(epochs,dtype=np.float64)
    for epoch in range(epochs):
        losses[epoch],grads=_numpy_mlp_gradients(X,y,params)
        params={k:params[k]-learning_rate*grads[k] for k in params}
    clean_scores=np.maximum(centers@params["W1"].T+params["b1"],0)@params["W2"].T+params["b2"]
    predictions=np.argmax(clean_scores,axis=1).astype(int)
    movement=max(float(np.linalg.norm(params[k]-initial[k])) for k in params)
    return {"params":params,"losses":losses,"predictions":predictions,"max_parameter_movement":movement}

result_p15=train_numpy_mlp(); repeat_p15=train_numpy_mlp()


### Answer check


In [ ]:
assert set(result_p15)=={"params","losses","predictions","max_parameter_movement"}
assert result_p15["losses"].shape==(800,) and np.all(np.isfinite(result_p15["losses"]))
assert result_p15["predictions"].shape==(4,) and np.mean(result_p15["predictions"]==np.array([0,1,1,0]))>=.95
assert result_p15["losses"][-1] < .35*result_p15["losses"][0]
assert np.allclose(result_p15["losses"],repeat_p15["losses"],atol=1e-10,rtol=1e-9)
assert all(np.allclose(result_p15["params"][k],repeat_p15["params"][k],atol=1e-10,rtol=1e-9) for k in result_p15["params"])

# Independently rebuild the prescribed first forward snapshot and all four gradients.
rng_check_p15=np.random.default_rng(20260804)
centers_check_p15=np.array([[-1.,-1.],[-1.,1.],[1.,-1.],[1.,1.]])
labels_check_p15=np.array([0,1,1,0])
X_check_p15=np.repeat(centers_check_p15,24,axis=0)+.12*rng_check_p15.standard_normal((96,2))
y_check_p15=np.repeat(labels_check_p15,24)
initial_p15={"W1":rng_check_p15.standard_normal((8,2))*np.sqrt(1/2),"b1":np.zeros(8),
             "W2":rng_check_p15.standard_normal((2,8))*np.sqrt(1/8),"b2":np.zeros(2)}
z1_check_p15=X_check_p15@initial_p15["W1"].T+initial_p15["b1"]
h_check_p15=np.maximum(z1_check_p15,0)
s_check_p15=h_check_p15@initial_p15["W2"].T+initial_p15["b2"]
e_check_p15=np.exp(s_check_p15-s_check_p15.max(1,keepdims=True))
p_check_p15=e_check_p15/e_check_p15.sum(1,keepdims=True)
ds_check_p15=p_check_p15.copy(); ds_check_p15[np.arange(96),y_check_p15]-=1; ds_check_p15/=96
expected_grads_p15={"W2":ds_check_p15.T@h_check_p15,"b2":ds_check_p15.sum(0)}
dz1_check_p15=(ds_check_p15@initial_p15["W2"])*(z1_check_p15>0)
expected_grads_p15.update(W1=dz1_check_p15.T@X_check_p15,b1=dz1_check_p15.sum(0))
_,actual_grads_p15=_numpy_mlp_gradients(X_check_p15,y_check_p15,initial_p15)
assert all(np.allclose(actual_grads_p15[k],expected_grads_p15[k],atol=1e-10,rtol=1e-9) for k in initial_p15)
movements_p15={k:float(np.linalg.norm(result_p15["params"][k]-initial_p15[k])) for k in initial_p15}
assert all(value>0.0 for value in movements_p15.values())
assert np.isclose(result_p15["max_parameter_movement"],max(movements_p15.values()),atol=1e-12,rtol=1e-12)
